# Scenarios

Each scenario copies `data/`, `sql/`, `run.py` and the current `output/` to a temporary folder, makes one change there
and runs the pipeline. The repo is never changed. The expected result is written before each run; the last section
sums up what happened.

- **A. Broken data:** the pipeline must block or warn, and never leave a wrong or stale number in `output/`.
- **B. Business requests:** a likely change needs only data, or one SQL file, and the numbers move as expected.

Run `python run.py` from the repo root first: the scenarios compare against the current `output/`.

## Setup

In [1]:
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path
from types import SimpleNamespace

import pandas as pd

REPO = Path("..").resolve()

# The current results, to compare each scenario with
base_allocation = pd.read_csv(REPO / "output" / "allocation.csv", dtype=str)
base_allocation["amount_usd"] = base_allocation["amount_usd"].astype(float)
base_reconciliation = pd.read_csv(REPO / "output" / "reconciliation.csv").set_index("period")
base_summary = pd.read_csv(REPO / "output" / "summary.csv")

summary = []

In [2]:
# Change one CSV of the copy: read it as text, apply the change, write it back
def edit(folder, name, change):
    path = folder / "data" / f"{name}.csv"
    table = pd.read_csv(path, dtype=str, keep_default_na=False)
    change(table).to_csv(path, index=False)


# Change one SQL file of the copy by replacing a piece of text
def edit_sql(folder, name, old, new):
    path = folder / "sql" / name
    text = path.read_text()
    assert old in text, f"{old!r} not found in {name}"
    path.write_text(text.replace(old, new))


# Copy the project to a temporary folder, apply the change, run the pipeline there
def run_scenario(change):
    folder = Path(tempfile.mkdtemp())
    for name in ["data", "sql", "output"]:
        shutil.copytree(REPO / name, folder / name)
    shutil.copy(REPO / "run.py", folder)
    change(folder)

    run = subprocess.run([sys.executable, "run.py"], cwd=folder, capture_output=True, text=True)
    output = folder / "output"
    files = sorted(p.name for p in output.glob("*.csv"))

    # The line that says why the run stopped (a blocked check or an error), else the last line
    lines = (run.stdout + run.stderr).strip().splitlines()
    stops = [line for line in lines if line.startswith("Blocked by") or "Error" in line]
    message = stops[-1] if stops else lines[-1]

    def read(name, **options):
        path = output / f"{name}.csv"
        return pd.read_csv(path, **options) if path.exists() else None

    result = SimpleNamespace(
        exit=run.returncode,
        files=files,
        message=message,
        checks=read("checks", index_col="check"),
        reconciliation=read("reconciliation", index_col="period"),
        allocation=read("allocation", dtype=str),
        summary=read("summary"),
    )
    if result.allocation is not None:
        result.allocation["amount_usd"] = result.allocation["amount_usd"].astype(float)
    shutil.rmtree(folder)  # everything needed is read; nothing is left behind
    print("exit:", result.exit, "| files:", files)
    print("message:", result.message)
    return result


# Keep one line per scenario for the summary at the end
def record(scenario, expected, ok):
    summary.append({"scenario": scenario, "expected": expected, "ok": bool(ok)})
    print("OK" if ok else "NOT AS EXPECTED")

## A. Broken data

### A1. A ledger line appears twice (Q2)

**Expected:** `block_duplicate_keys` fails, exit 1, only `checks.csv` left in `output/`.

In [3]:
r = run_scenario(lambda f: edit(f, "ledger", lambda t: pd.concat([t, t.head(1)])))
print(r.checks["result"])
record("A1 duplicate ledger line", "block_duplicate_keys fails, only checks.csv",
       r.exit == 1 and r.checks.loc["block_duplicate_keys", "result"] == "fail" and r.files == ["checks.csv"])

exit: 1 | files: ['checks.csv']
message: Blocked by: block_duplicate_keys, block_reconciliation, block_shares_not_one. No results written; see output/checks.csv
check
block_allocated_rows_complete    pass
block_duplicate_keys             fail
block_months_outside_quarter     pass
block_overlapping_mapping        pass
block_reconciliation             fail
block_shares_not_one             fail
block_unmatched_game_names       pass
warn_trimmed_values              warn
warn_unallocated                 warn
Name: result, dtype: str
OK


### A2. Two mapping rows valid in the same month (Q3)

**Expected:** `block_overlapping_mapping` fails, exit 1, only `checks.csv` left.

In [4]:
new_row = {"game_group": "Pixel Farm", "category": "New Games", "bearing_group": "", "side": "BR Games",
           "start_date": "2026-01-01", "end_date": "9999-12-31"}
r = run_scenario(lambda f: edit(f, "game_mapping", lambda t: pd.concat([t, pd.DataFrame([new_row])])))
record("A2 overlapping mapping rows", "block_overlapping_mapping fails, only checks.csv",
       r.exit == 1 and r.checks.loc["block_overlapping_mapping", "result"] == "fail" and r.files == ["checks.csv"])

exit: 1 | files: ['checks.csv']
message: Blocked by: block_overlapping_mapping, block_reconciliation, block_shares_not_one. No results written; see output/checks.csv
OK


### A3. A ledger line in April (Q9)

**Expected:** `block_months_outside_quarter` fails, exit 1, only `checks.csv` left.

In [5]:
def april(t):
    t.loc[t["line_id"] == "L00105", "month"] = "2026-04"
    return t

r = run_scenario(lambda f: edit(f, "ledger", april))
record("A3 month outside the quarter", "block_months_outside_quarter fails, only checks.csv",
       r.exit == 1 and r.checks.loc["block_months_outside_quarter", "result"] == "fail" and r.files == ["checks.csv"])

exit: 1 | files: ['checks.csv']
message: Blocked by: block_months_outside_quarter. No results written; see output/checks.csv
OK


### A4. A date the load step can't read (decision 4)

**Expected:** the load step stops with an error, exit 1, and `output/` is empty: no stale result from the previous run.

In [6]:
def bad_date(t):
    t.loc[0, "start_date"] = "01/03/2026"
    return t

r = run_scenario(lambda f: edit(f, "game_mapping", bad_date))
record("A4 unreadable date", "error in the load step, output/ empty", r.exit == 1 and r.files == [])

exit: 1 | files: []
message: Error when converting column "start_date". Could not convert string "01/03/2026" to 'DATE'
OK


### A5. A ledger line with an unknown entity (Q1)

**Expected:** L00105 (103,255.10) is unallocated with `unknown_entity`; the run warns and reconciles.

In [7]:
def unknown_entity(t):
    t.loc[t["line_id"] == "L00105", "entity"] = "LGX"
    return t

r = run_scenario(lambda f: edit(f, "ledger", unknown_entity))
row = r.allocation[r.allocation["line_id"] == "L00105"]
print(row[["line_id", "booked_entity", "booked_group", "amount_usd", "status", "reason"]])
record("A5 unknown entity", "L00105 unallocated, unknown_entity, reconciles",
       r.exit == 0 and row["reason"].tolist() == ["unknown_entity"]
       and r.reconciliation.loc["2026-01", "unallocated"] == 103255.10)

exit: 0 | files: ['allocation.csv', 'checks.csv', 'reconciliation.csv', 'summary.csv']
message: wrote output/reconciliation.csv
    line_id booked_entity booked_group  amount_usd       status  \
361  L00105           LGX          NaN    103255.1  unallocated   

             reason  
361  unknown_entity  
OK


### A6. A ledger line with an unknown account (Q1)

**Expected:** L00105 is unallocated with `unknown_account` and counts as in scope; in-scope dollars don't change.

In [8]:
def unknown_account(t):
    t.loc[t["line_id"] == "L00105", "account"] = "9999"
    return t

r = run_scenario(lambda f: edit(f, "ledger", unknown_account))
row = r.allocation[r.allocation["line_id"] == "L00105"]
print(row[["line_id", "account", "account_group", "amount_usd", "status", "reason"]])
record("A6 unknown account", "L00105 unallocated, unknown_account, in scope unchanged",
       r.exit == 0 and row["reason"].tolist() == ["unknown_account"]
       and r.reconciliation.loc["2026-Q1", "in_scope"] == base_reconciliation.loc["2026-Q1", "in_scope"])

exit: 0 | files: ['allocation.csv', 'checks.csv', 'reconciliation.csv', 'summary.csv']
message: wrote output/reconciliation.csv
    line_id account account_group  amount_usd       status           reason
361  L00105    9999           NaN    103255.1  unallocated  unknown_account
OK


### A7. `Depreciation ` with a trailing space (Q8)

**Expected:** the trim is logged, and the 12 Depreciation lines stay out of scope (98,044.91).

In [9]:
def depreciation_space(t):
    t.loc[t["account_group"] == "Depreciation", "account_group"] = "Depreciation "
    return t

r = run_scenario(lambda f: edit(f, "accounts", depreciation_space))
print(r.checks.loc["warn_trimmed_values"])
record("A7 Depreciation with a trailing space", "trim logged, out of scope unchanged",
       r.exit == 0 and r.checks.loc["warn_trimmed_values", "rows"] == 2
       and r.reconciliation.loc["2026-Q1", "out_of_scope"] == base_reconciliation.loc["2026-Q1", "out_of_scope"])

exit: 0 | files: ['allocation.csv', 'checks.csv', 'reconciliation.csv', 'summary.csv']
message: wrote output/reconciliation.csv
level         warn
result        warn
rows             2
amount_usd     NaN
Name: warn_trimmed_values, dtype: object
OK


### A8. A pool with no FTE in a month (R5, Q1)

**Expected:** every in-scope LIVE_SHARED line of February has one row, `nothing_to_split`; the run warns and reconciles.

In [10]:
live_shared = ["Star Racer", "Kingdom Clash", "Ocean Tales", "Legacy Arcade"]
r = run_scenario(lambda f: edit(f, "headcount_games",
                                lambda t: t[~((t["month"] == "2026-02") & t["game"].isin(live_shared))]))

pool_lines = r.allocation[(r.allocation["pool_id"] == "LIVE_SHARED") & (r.allocation["month"] == "2026-02")
                          & (r.allocation["status"] != "out_of_scope")]
print(pool_lines["reason"].value_counts(), "| lines:", pool_lines["line_id"].nunique(), "| rows:", len(pool_lines))
record("A8 pool with no FTE", "February in-scope LIVE_SHARED lines: nothing_to_split, one row each",
       r.exit == 0 and set(pool_lines["reason"]) == {"nothing_to_split"}
       and len(pool_lines) == pool_lines["line_id"].nunique()
       and r.checks.loc["block_reconciliation", "result"] == "pass")

exit: 0 | files: ['allocation.csv', 'checks.csv', 'reconciliation.csv', 'summary.csv']
message: wrote output/reconciliation.csv
reason
nothing_to_split    20
Name: count, dtype: int64 | lines: 20 | rows: 20
OK


### A9. An employee with an unknown entity (Notes)

**Expected:** their bucket is `live_unknown`, with no mapping row: a piece of every January indirect line is unallocated, `no_mapping_row`.

In [11]:
def unknown_employee_entity(t):
    t.loc[(t["month"] == "2026-01") & (t["employee_id"] == "E0001"), "entity"] = "LGX"
    return t

r = run_scenario(lambda f: edit(f, "headcount_employees", unknown_employee_entity))
pieces = r.allocation[r.allocation["game_or_bucket"] == "live_unknown"]
print(pieces["reason"].value_counts(), "| unallocated in January:", r.reconciliation.loc["2026-01", "unallocated"])
record("A9 employee with an unknown entity", "live_unknown pieces, no_mapping_row, reconciles",
       r.exit == 0 and len(pieces) > 0 and set(pieces["reason"]) == {"no_mapping_row"}
       and r.checks.loc["block_reconciliation", "result"] == "pass")

exit: 0 | files: ['allocation.csv', 'checks.csv', 'reconciliation.csv', 'summary.csv']
message: wrote output/reconciliation.csv
reason
no_mapping_row    83
Name: count, dtype: int64 | unallocated in January: 7959.58
OK


### A10. A direct cost center with both a game and a pool (Q1)

**Expected:** every line on cost center 1001 (Star Racer) is unallocated with `bad_cost_center`.

In [12]:
def game_and_pool(t):
    t.loc[t["cost_center"] == "1001", "pool_id"] = "LIVE_SHARED"
    return t

r = run_scenario(lambda f: edit(f, "cost_centers", game_and_pool))
rows = r.allocation[r.allocation["cost_center"] == "1001"]
print(rows["reason"].value_counts())
record("A10 cost center with game and pool", "lines on 1001: bad_cost_center",
       r.exit == 0 and set(rows["reason"]) == {"bad_cost_center"})

exit: 0 | files: ['allocation.csv', 'checks.csv', 'reconciliation.csv', 'summary.csv']
message: wrote output/reconciliation.csv
reason
bad_cost_center    20
Name: count, dtype: int64
OK


### A11. A misspelled game in headcount_games.csv (found in the adversarial review)

**Expected:** `block_unmatched_game_names` fails, exit 1, only `checks.csv` left. Without that check, Pixel Farm's January FTE would drop out of its pools and its dollars would move to the other pool games, with every other check passing.

In [13]:
def misspell(t):
    t.loc[(t["month"] == "2026-01") & (t["game"] == "Pixel Farm"), "game"] = "Pixel farm"
    return t

r = run_scenario(lambda f: edit(f, "headcount_games", misspell))
record("A11 misspelled game in headcount", "block_unmatched_game_names fails, only checks.csv",
       r.exit == 1 and r.checks.loc["block_unmatched_game_names", "result"] == "fail" and r.files == ["checks.csv"])

exit: 1 | files: ['checks.csv']
message: Blocked by: block_unmatched_game_names. No results written; see output/checks.csv
OK


## B. Business requests

### B1. "Cost center 3999 is Marketing" (D1)

**Change:** data only, one row in `cost_centers.csv`.

**Expected:** L00169 is split across February's buckets; February has nothing unallocated.

In [14]:
new_row = {"cost_center": "3999", "cost_center_name": "Marketing (3999)", "department": "Marketing",
           "game": "", "pool_id": ""}
r = run_scenario(lambda f: edit(f, "cost_centers", lambda t: pd.concat([t, pd.DataFrame([new_row])])))
rows = r.allocation[r.allocation["line_id"] == "L00169"]
print(rows[["line_id", "game_or_bucket", "amount_usd", "side", "status"]])
record("B1 3999 is Marketing", "L00169 allocated, February unallocated = 0",
       r.exit == 0 and set(rows["status"]) == {"allocated"} and r.reconciliation.loc["2026-02", "unallocated"] == 0)

exit: 0 | files: ['allocation.csv', 'checks.csv', 'reconciliation.csv', 'summary.csv']
message: wrote output/reconciliation.csv
    line_id game_or_bucket  amount_usd      side     status
560  L00169        live_br     9823.39  BR Games  allocated
561  L00169         new_br     3458.94  BR Games  allocated
562  L00169         new_fi     1106.86  US Games  allocated
563  L00169     new_others      691.79  US Games  allocated
564  L00169         new_us     1937.01  US Games  allocated
OK


### B2. "Add the mapping row for live_fi" (D3)

**Change:** data only, one row in `game_mapping.csv`. The side BR Games follows `live_others`; it's an assumption for this scenario.

**Expected:** March has nothing unallocated.

In [15]:
new_row = {"game_group": "live_fi", "category": "Live Games", "bearing_group": "FI", "side": "BR Games",
           "start_date": "2024-01-01", "end_date": "9999-12-31"}
r = run_scenario(lambda f: edit(f, "game_mapping", lambda t: pd.concat([t, pd.DataFrame([new_row])])))
print(r.reconciliation["unallocated"])
record("B2 live_fi mapping row", "March unallocated = 0",
       r.exit == 0 and r.reconciliation.loc["2026-03", "unallocated"] == 0)

exit: 0 | files: ['allocation.csv', 'checks.csv', 'reconciliation.csv', 'summary.csv']
message: wrote output/reconciliation.csv
period
2026-01        0.00
2026-02    17017.98
2026-03        0.00
2026-Q1    17017.98
Name: unallocated, dtype: float64
OK


### B3. "Puzzle Pals moved to US Games in February, not March" (R10)

**Change:** data only, the dates of the two Puzzle Pals rows.

**Expected:** Puzzle Pals' February dollars move from BR Games to US Games; totals don't change.

In [16]:
def move_in_february(t):
    t.loc[(t["game_group"] == "Puzzle Pals") & (t["side"] == "BR Games"), "end_date"] = "2026-01-31"
    t.loc[(t["game_group"] == "Puzzle Pals") & (t["side"] == "US Games"), "start_date"] = "2026-02-01"
    return t

r = run_scenario(lambda f: edit(f, "game_mapping", move_in_february))

def puzzle_pals_february(allocation):
    rows = allocation[(allocation["game_or_bucket"] == "Puzzle Pals") & (allocation["month"] == "2026-02")]
    return rows.groupby("side")["amount_usd"].sum().round(2)

print("before:", puzzle_pals_february(base_allocation).to_dict())
print("after: ", puzzle_pals_february(r.allocation).to_dict())
record("B3 Puzzle Pals moves in February", "February Puzzle Pals dollars move to US Games",
       r.exit == 0 and list(puzzle_pals_february(r.allocation).index) == ["US Games"]
       and r.reconciliation.loc["2026-02", "allocated"] == base_reconciliation.loc["2026-02", "allocated"])

exit: 0 | files: ['allocation.csv', 'checks.csv', 'reconciliation.csv', 'summary.csv']
message: wrote output/reconciliation.csv
before: {'BR Games': 97871.3}
after:  {'US Games': 97871.3}
OK


### B4. "Also exclude account group Administrative" (R1)

**Change:** one SQL file, `03_lines.sql`: a new reason and its place in the out-of-scope list.

**Expected:** out of scope grows by the Administrative in-scope dollars; the run still reconciles.

In [17]:
def exclude_administrative(f):
    edit_sql(f, "03_lines.sql",
             "WHEN account_group = 'Depreciation' THEN 'depreciation'",
             "WHEN account_group = 'Depreciation' THEN 'depreciation'\n"
             "            WHEN account_group = 'Administrative' THEN 'administrative'")
    edit_sql(f, "03_lines.sql", "('zero_amount', 'depreciation')", "('zero_amount', 'depreciation', 'administrative')")

r = run_scenario(exclude_administrative)
# The Administrative dollars, from the ledger itself (line amounts, not rounded pieces)
ledger = pd.read_csv(REPO / "data" / "ledger.csv", dtype={"account": str})
accounts = pd.read_csv(REPO / "data" / "accounts.csv", dtype={"account": str})
administrative = ledger.merge(accounts, on="account").query("account_group == 'Administrative'")["amount_usd"].sum()
growth = r.reconciliation.loc["2026-Q1", "out_of_scope"] - base_reconciliation.loc["2026-Q1", "out_of_scope"]
print("Administrative in scope before:", round(administrative, 2), "| out of scope grew by:", round(growth, 2))
record("B4 exclude Administrative", "out of scope grows by the Administrative dollars, reconciles",
       r.exit == 0 and abs(growth - administrative) < 0.005 and r.checks.loc["block_reconciliation", "result"] == "pass")

exit: 0 | files: ['allocation.csv', 'checks.csv', 'reconciliation.csv', 'summary.csv']
message: wrote output/reconciliation.csv
Administrative in scope before: 802623.33 | out of scope grew by: 802623.33
OK


### B5. "A new game with its own cost center" (R4, R7)

**Change:** data only: one cost center, one mapping row, and one ledger line moved to it.

**Expected:** the line goes 100% to the new game with the side from its mapping row; no SQL changes.

In [18]:
def new_game(f):
    edit(f, "cost_centers", lambda t: pd.concat([t, pd.DataFrame([{
        "cost_center": "2006", "cost_center_name": "Star Chef", "department": "New Games", "game": "Star Chef", "pool_id": ""}])]))
    edit(f, "game_mapping", lambda t: pd.concat([t, pd.DataFrame([{
        "game_group": "Star Chef", "category": "New Games", "bearing_group": "", "side": "US Games",
        "start_date": "2026-01-01", "end_date": "9999-12-31"}])]))

    def move_line(t):
        t.loc[t["line_id"] == "L00105", "cost_center"] = "2006"
        return t
    edit(f, "ledger", move_line)

r = run_scenario(new_game)
row = r.allocation[r.allocation["line_id"] == "L00105"]
print(row[["line_id", "cost_center", "game_or_bucket", "share", "amount_usd", "side", "status"]])
record("B5 new game", "L00105 allocated 100% to Star Chef, US Games",
       r.exit == 0 and row["game_or_bucket"].tolist() == ["Star Chef"] and row["side"].tolist() == ["US Games"])

exit: 0 | files: ['allocation.csv', 'checks.csv', 'reconciliation.csv', 'summary.csv']
message: wrote output/reconciliation.csv
    line_id cost_center game_or_bucket share  amount_usd      side     status
361  L00105        2006      Star Chef   1.0    103255.1  US Games  allocated
OK


### B6. A live edit breaks the pool split (scenario S8)

**Change:** one SQL file, `05_direct_pool.sql`: the FTE total is taken per pool instead of per pool and month.

**Expected:** `block_shares_not_one` fails, exit 1, the last line names it, and the old results are gone.

In [19]:
r = run_scenario(lambda f: edit_sql(f, "05_direct_pool.sql",
                                   "PARTITION BY p.pool_id, h.month", "PARTITION BY p.pool_id"))
print(r.checks["result"])
record("B6 broken pool edit", "block_shares_not_one fails, named in the last line, only checks.csv",
       r.exit == 1 and r.checks.loc["block_shares_not_one", "result"] == "fail"
       and "block_shares_not_one" in r.message and r.files == ["checks.csv"])

exit: 1 | files: ['checks.csv']
message: Blocked by: block_reconciliation, block_shares_not_one. No results written; see output/checks.csv
check
block_allocated_rows_complete    pass
block_duplicate_keys             pass
block_months_outside_quarter     pass
block_overlapping_mapping        pass
block_reconciliation             fail
block_shares_not_one             fail
block_unmatched_game_names       pass
warn_trimmed_values              warn
warn_unallocated                 warn
Name: result, dtype: str
OK


### B7. "Split indirect costs only among the booking entity's employees" (the other reading of Q4)

**Change:** one SQL file, `06_indirect.sql`: bucket shares per month and entity, joined on the line's entity.

**Expected:** indirect costs stay in the booking entity's group (bearing group = booked group on every indirect row); the totals by side change; the run reconciles. The numbers go to the Tax note as the impact of this assumption.

In [20]:
def own_entity_only(f):
    edit_sql(f, "06_indirect.sql", "    h.month,\n    lower(", "    h.month,\n    h.entity,\n    lower(")
    edit_sql(f, "06_indirect.sql",
             "    month,\n    bucket,\n    COUNT(*)::DOUBLE / SUM(COUNT(*)) OVER (PARTITION BY month) AS share\nFROM employee_buckets\nGROUP BY month, bucket;",
             "    month,\n    entity,\n    bucket,\n    COUNT(*)::DOUBLE / SUM(COUNT(*)) OVER (PARTITION BY month, entity) AS share\nFROM employee_buckets\nGROUP BY month, entity, bucket;")
    edit_sql(f, "06_indirect.sql", "JOIN bucket_shares s ON s.month = l.month",
             "JOIN bucket_shares s ON s.month = l.month AND s.entity = l.booked_entity")

r = run_scenario(own_entity_only)

def quarter_by_side(summary):
    return summary[summary["period"].str.contains("Q")][["br_games", "us_games", "unallocated"]].sum().round(2)

change = pd.DataFrame({"as built (whole company)": quarter_by_side(base_summary),
                       "booking entity only": quarter_by_side(r.summary)})
change["difference"] = change["booking entity only"] - change["as built (whole company)"]
print(change)
# In-scope dollars don't change, so the differences net to zero. Up to a cent can show, because summary.csv cells
# are rounded; in whole dollars they net to exactly zero.
print("net:", round(change["difference"].sum(), 2), "| in whole dollars:", change["difference"].round(0).to_dict())

indirect = r.allocation[(r.allocation["cost_type"] == "indirect") & (r.allocation["status"] == "allocated")]
record("B7 Q4 read as booking entity only", "indirect costs stay in their booked group; reconciles",
       r.exit == 0 and (indirect["bearing_group"] == indirect["booked_group"]).all()
       and r.checks.loc["block_reconciliation", "result"] == "pass")

exit: 0 | files: ['allocation.csv', 'checks.csv', 'reconciliation.csv', 'summary.csv']
message: wrote output/reconciliation.csv
             as built (whole company)  booking entity only  difference
br_games                   4642414.76           4461434.92  -180979.84
us_games                   1864780.12           2053624.93   188844.81
unallocated                  32055.97             24191.01    -7864.96
net: 0.01 | in whole dollars: {'br_games': -180980.0, 'us_games': 188845.0, 'unallocated': -7865.0}
OK


## Summary

In [21]:
pd.DataFrame(summary)

,scenario,expected,ok
0,A1 duplicate ledger line,"block_duplicate_keys fails, only checks.csv",True
1,A2 overlapping mapping rows,"block_overlapping_mapping fails, only checks.csv",True
2,A3 month outside the quarter,"block_months_outside_quarter fails, only check...",True
3,A4 unreadable date,"error in the load step, output/ empty",True
4,A5 unknown entity,"L00105 unallocated, unknown_entity, reconciles",True
5,A6 unknown account,"L00105 unallocated, unknown_account, in scope ...",True
6,A7 Depreciation with a trailing space,"trim logged, out of scope unchanged",True
7,A8 pool with no FTE,February in-scope LIVE_SHARED lines: nothing_t...,True
8,A9 employee with an unknown entity,"live_unknown pieces, no_mapping_row, reconciles",True
9,A10 cost center with game and pool,lines on 1001: bad_cost_center,True


**R:** All 18 scenarios behave as expected. Every broken input blocks or warns, and no blocked run leaves a result in `output/`. Every business request needed only data or one SQL file, and the run still reconciled.